# Exploração do dataset de Energia

Este notebook tem como objetivo caracterizar os padrões de consumo energético por ponto de entrega (CPE), com foco na identificação de padrões temporais, contextuais e sazonais relevantes para análise posterior.

A exploração dos dados constitui uma etapa fundamental para compreender o comportamento das séries temporais e suportar fases seguintes, nomeadamente a extração de features, clustering e deteção de anomalias.

Em particular, pretende-se:

- analisar os 91 pontos de consumo (CPE);
- estudar perfis médios diários por hora;
- observar perfis reais ao longo do tempo, sem agregação;
- distinguir períodos de utilização e não utilização;
- analisar efeitos de calendário (dias úteis, fins de semana, sazonalidade e dias especiais);
- caracterizar o consumo em diferentes escalas temporais (diária e mensal);
- extrair indicadores simples que permitam descrever os perfis de consumo.

A variável principal considerada ao longo do estudo é **PotActiva**, utilizada como proxy do consumo energético.

# Preparação e caracterização inicial

In [ ]:
# Imports e configuração

import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Configuração de visualização
plt.rcParams["figure.figsize"] = (12, 4)
sns.set_theme(style="whitegrid")

base_dir = Path("../results")

# Pastas principais
exploration_dir = base_dir / "exploration"
features_dir = base_dir / "features"
clustering_dir = base_dir / "clustering"
anomalies_dir = base_dir / "anomalies"
models_dir = base_dir / "models"

# Subpastas específicas
intermediate_dir = exploration_dir / "intermediate"

# Clustering
clustering_analysis_dir = clustering_dir / "analysis"
clustering_plots_dir = clustering_dir / "plots"

# Features
features_plots_dir = features_dir / "plots"
features_validation_dir = features_dir / "validation"

# Anomalies
anomalies_plots_dir = anomalies_dir / "plots"
anomalies_temporal_plots_dir = anomalies_dir / "anomalies_temporal_plots"

for d in [
    exploration_dir,
    features_dir,
    clustering_dir,
    anomalies_dir,
    models_dir,
    intermediate_dir,
    clustering_analysis_dir,
    clustering_plots_dir,
    features_plots_dir,
    features_validation_dir,
    anomalies_plots_dir,
    anomalies_temporal_plots_dir,
]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load do dataset

repo_datasets_path = Path("../PM2025-TSAnomalyDetection/example-timeseries")
zip_path = repo_datasets_path / "consumo15m_11_2025.csv.zip"

# Verificação da existência dos ficheiros
assert repo_datasets_path.exists(), f"Caminho não encontrado: {repo_datasets_path}"
assert zip_path.exists(), f"Ficheiro não encontrado: {zip_path}"

with zipfile.ZipFile(zip_path) as z:
    print("Conteúdo do ZIP:", z.namelist())
    
    with z.open(z.namelist()[0]) as f:
        df_energia = pd.read_csv(f)

df_energia.head()

In [ ]:
# Preparação inicial

df_energia["tstamp"] = pd.to_datetime(df_energia["tstamp"])
df_energia = df_energia.sort_values(["CPE", "tstamp"]).reset_index(drop=True)

print("Shape do dataset:", df_energia.shape)

In [ ]:
# Estrutura do dataset

print("Informação geral:")
display(df_energia.info())

print("\nColunas disponíveis:")
display(df_energia.columns)

print("\nNúmero de CPE distintos:", df_energia["CPE"].nunique())
print("Início:", df_energia["tstamp"].min())
print("Fim:", df_energia["tstamp"].max())

In [ ]:
# Qualidade dos dados

# Número de valores em falta por coluna
missing_values = df_energia.isna().sum().sort_values(ascending=False)

# Percentagem de missing values
missing_pct = (df_energia.isna().mean() * 100).sort_values(ascending=False)

quality_df = pd.DataFrame({
    "missing_count": missing_values,
    "missing_pct": missing_pct.round(4)
})

quality_df

In [ ]:
# Escolha da variável principal

df_energia["PotActiva"].describe()

In [ ]:
# Pivot table por CPE

df_pivot = df_energia.pivot_table(
    index="tstamp",
    columns="CPE",
    values="PotActiva"
)

print("Shape da pivot table:", df_pivot.shape)

df_pivot.head()

In [ ]:
# Regularidade temporal global

time_diffs = df_pivot.index.to_series().diff().value_counts().sort_values(ascending=False)

print("Diferenças temporais mais frequentes:")
display(time_diffs.head(10))

# Frequência mais comum
freq_dominante = df_pivot.index.to_series().diff().value_counts().idxmax()
print("Frequência temporal dominante:", freq_dominante)

In [ ]:
# Missing values por CPE

missing_pct_por_cpe = (df_pivot.isna().mean() * 100).sort_values()

print("CPE com menor percentagem de missing:")
display(missing_pct_por_cpe.head(10))

print("Resumo da percentagem de missing por CPE:")
display(missing_pct_por_cpe.describe())

In [ ]:
# Distribuição de missing values por CPE

plt.figure(figsize=(10, 4))
plt.hist(missing_pct_por_cpe, bins=30)
plt.title("Distribuição da percentagem de missing values por CPE")
plt.xlabel("Missing (%)")
plt.ylabel("Frequência")
plt.tight_layout()
plt.show()

In [ ]:
# Consumo médio por CPE

consumo_medio_por_cpe = df_pivot.mean().sort_values(ascending=False)

print("Top 10 CPE por consumo médio de PotActiva:")
display(consumo_medio_por_cpe.head(10))

print("Resumo do consumo médio por CPE:")
display(consumo_medio_por_cpe.describe())

## Observações

A análise realizada permite retirar várias conclusões relevantes sobre o dataset:

- O conjunto de dados inclui múltiplas séries temporais (91 CPE), evidenciando a necessidade de uma abordagem individualizada.
- A frequência dominante de 15 minutos garante uma elevada resolução temporal, adequada para análise detalhada dos padrões de consumo.
- Verifica-se a presença de valores em falta, com elevada variabilidade entre CPE, o que poderá impactar etapas posteriores.
- A variável **PotActiva** apresenta-se como uma medida representativa do consumo energético e adequada para análise.
- A existência de diferenças significativas no consumo médio entre CPE sugere a presença de múltiplos perfis de utilização.

Estas observações reforçam a necessidade de aprofundar a análise ao nível dos padrões temporais, contexto de utilização e sazonalidade, explorados nas secções seguintes.

# Perfis médios diários por hora

Nesta secção, é analisado o comportamento médio do consumo energético ao longo das 24 horas do dia.

Para cada CPE, é calculado o perfil médio horário, agregando os valores de consumo por hora do dia.

Esta abordagem permite identificar padrões típicos de utilização, suavizando a variabilidade diária e evidenciando tendências estruturais no comportamento dos diferentes pontos de consumo.

In [ ]:
# Perfil médio diário por hora

df_hourly = df_pivot.copy()
df_hourly["hora"] = df_hourly.index.hour

# Cálculo do perfil médio por hora
perfil_horario = df_hourly.groupby("hora").mean()

print("Shape do perfil horário:", perfil_horario.shape)
perfil_horario.head()

In [ ]:
# Visualização por CPE

output_dir = Path("../results")

graficos_dir = exploration_dir / "perfis_horarios_cpe"
graficos_dir.mkdir(parents=True, exist_ok=True)

for cpe in perfil_horario.columns:

    plt.figure(figsize=(8, 4))
    plt.plot(perfil_horario.index, perfil_horario[cpe], marker="o")
    plt.title(f"Perfil médio diário - {cpe}")
    plt.xlabel("Hora do dia")
    plt.ylabel("PotActiva média")
    plt.xticks(range(0, 24))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(graficos_dir / f"{cpe}_perfil_diario.png", dpi=150)
    plt.show()
    plt.close()

print(f"Gráficos guardados em: {graficos_dir}")

## Observações

A análise dos perfis médios diários permite identificar diferentes padrões de consumo energético entre os CPE, evidenciando a existência de múltiplos comportamentos característicos.

Em particular, observam-se:

- perfis com consumo predominantemente diurno, típicos de edifícios com atividade laboral;
- perfis com consumo mais distribuído ou com atividade noturna, possivelmente associados a sistemas automáticos ou funcionamento contínuo;
- perfis com picos acentuados em horários específicos, indicando utilização intermitente;
- perfis híbridos, combinando diferentes padrões ao longo do dia.

A diversidade de comportamentos observados confirma a heterogeneidade dos pontos de consumo, reforçando a necessidade de uma análise individualizada e justificando a aplicação de técnicas de clustering em fases posteriores.

Importa, no entanto, destacar que estes perfis resultam de uma agregação por média, o que implica perda de informação temporal. Assim, embora sejam úteis para identificar padrões gerais, não capturam variações específicas entre dias.

Por este motivo, na secção seguinte são analisados perfis reais ao longo do tempo, preservando a estrutura temporal original das séries.

# Perfis reais ao longo do tempo (Março 2025)

Após a análise dos perfis médios, é fundamental observar o comportamento real das séries temporais sem qualquer agregação.

Nesta secção, é analisado o consumo energético ao longo do mês de março de 2025, permitindo preservar a estrutura temporal original dos dados.

Esta abordagem possibilita identificar variações intra-diárias e inter-diárias, bem como padrões que não são visíveis em perfis médios, sendo particularmente relevante para a deteção de anomalias.

In [ ]:
# Filtrar março de 2025

df_marco = df_energia[
    (df_energia["tstamp"].dt.year == 2025) &
    (df_energia["tstamp"].dt.month == 3)
].copy()

print("Shape março:", df_marco.shape)
print("Número de CPE em março:", df_marco["CPE"].nunique())
print("Início março:", df_marco["tstamp"].min())
print("Fim março:", df_marco["tstamp"].max())

In [ ]:
# Visualização por CPE

graficos_dir = exploration_dir / "perfis_reais_marco_cpe"
graficos_dir.mkdir(parents=True, exist_ok=True)

for cpe in df_marco["CPE"].unique():

    df_cpe = df_marco[df_marco["CPE"] == cpe].sort_values("tstamp")

    plt.figure(figsize=(14, 4))
    plt.plot(df_cpe["tstamp"], df_cpe["PotActiva"])
    plt.title(f"{cpe} — Perfil real em Março 2025")
    plt.xlabel("Tempo")
    plt.ylabel("PotActiva")
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=2))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(graficos_dir / f"{cpe}_perfil_real_marco_2025.png", dpi=150)
    plt.show()
    plt.close()

print(f"Gráficos individuais guardados em: {graficos_dir}")

## Observações

Ao contrário dos perfis médios anteriormente analisados, estes gráficos permitem observar diretamente o comportamento real das séries temporais, preservando a sua dinâmica temporal completa.

A análise evidencia:

- a existência de ciclos diários consistentes, refletindo padrões regulares de utilização;
- variações significativas entre dias consecutivos, indicando que o consumo não é totalmente estável;
- períodos de maior e menor atividade, associados a diferentes contextos de utilização;
- a presença de picos, quebras e possíveis interrupções, que poderão corresponder a eventos específicos ou comportamentos anómalos.

Esta abordagem é particularmente relevante no contexto de deteção de anomalias, uma vez que permite identificar desvios face ao comportamento esperado.

Adicionalmente, a observação direta das séries reforça a ideia de que o comportamento dos CPE não pode ser totalmente capturado por métricas agregadas, justificando a utilização de métodos que preservem a estrutura temporal dos dados.

# Utilização VS Não Utilização

Nesta secção, é introduzido um mecanismo para distinguir automaticamente períodos de utilização e não utilização para cada CPE, com base no comportamento real de consumo.

Ao contrário de abordagens baseadas apenas no calendário (ex: dias úteis vs fins de semana), esta metodologia adapta-se ao perfil específico de cada ponto de consumo, permitindo uma caracterização mais realista dos padrões de funcionamento.

Esta distinção constitui uma forma de contextualização dos dados, sendo particularmente relevante para a extração de features e para a deteção de anomalias.

In [ ]:
# Construção de variáveis contextuais

df_contexto = df_energia.copy()

# Componentes temporais
df_contexto["data"] = df_contexto["tstamp"].dt.date
df_contexto["hora"] = df_contexto["tstamp"].dt.hour
df_contexto["dia_semana"] = df_contexto["tstamp"].dt.dayofweek

In [ ]:
# Classificação de utilização por CPE

# Consumo médio diário por CPE
consumo_diario_cpe = (
    df_contexto
    .groupby(["CPE", "data"])["PotActiva"]
    .mean()
    .reset_index()
)

# Média global por CPE
consumo_diario_cpe["mean_cpe"] = (
    consumo_diario_cpe
    .groupby("CPE")["PotActiva"]
    .transform("mean")
)

# Classificação baseada em threshold relativo
consumo_diario_cpe["tipo_utilizacao"] = np.where(
    consumo_diario_cpe["PotActiva"] > 0.3 * consumo_diario_cpe["mean_cpe"],
    "utilizacao",
    "nao_utilizacao"
)

consumo_diario_cpe.head()

In [ ]:
# Integração no dataset principal

df_contexto = df_contexto.merge(
    consumo_diario_cpe[["CPE", "data", "tipo_utilizacao"]],
    on=["CPE", "data"],
    how="left"
)

df_contexto[["tstamp", "CPE", "tipo_utilizacao"]].head()

In [ ]:
# Resumo

print("Distribuição por tipo de utilização:")
display(df_contexto["tipo_utilizacao"].value_counts())

Após a classificação, são analisados os perfis médios horários separadamente para períodos de utilização e não utilização.

Esta comparação permite identificar diferenças estruturais no comportamento do consumo energético.

In [ ]:
# Perfil médio horário por tipo de utilização

perfil_utilizacao = (
    df_contexto
    .groupby(["CPE", "tipo_utilizacao", "hora"])["PotActiva"]
    .mean()
    .reset_index()
)

perfil_utilizacao.head()

In [ ]:
# Visualização por CPE

graficos_dir = exploration_dir / "perfis_utilizacao_vs_nao_utilizacao"
graficos_dir.mkdir(parents=True, exist_ok=True)

for cpe in perfil_utilizacao["CPE"].unique():
    df_cpe = perfil_utilizacao[perfil_utilizacao["CPE"] == cpe]

    plt.figure(figsize=(8, 4))

    for tipo in df_cpe["tipo_utilizacao"].unique():
        df_tipo = df_cpe[df_cpe["tipo_utilizacao"] == tipo]
        plt.plot(df_tipo["hora"], df_tipo["PotActiva"], marker="o", label=tipo)

    plt.title(f"{cpe} - Utilização vs Não Utilização")
    plt.xlabel("Hora do dia")
    plt.ylabel("PotActiva média")
    plt.xticks(range(0, 24))
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(graficos_dir / f"{cpe}_utilizacao_vs_nao_utilizacao.png", dpi=150)
    plt.show()
    plt.close()

print(f"Gráficos guardados em: {graficos_dir}")

In [ ]:
# Medida quantitativa da diferença

perfil_utilizacao_wide = (
    perfil_utilizacao
    .pivot_table(index=["CPE", "hora"], columns="tipo_utilizacao", values="PotActiva")
    .reset_index()
)

diff_utilizacao = (
    perfil_utilizacao_wide
    .assign(diff=lambda x: (x["utilizacao"] - x["nao_utilizacao"]).abs())
    .groupby("CPE")["diff"]
    .mean()
    .sort_values(ascending=False)
)

print("CPE com maior diferença entre utilização e não utilização:")
display(diff_utilizacao.head(10))

## Observações

A classificação baseada no consumo real permite distinguir de forma adaptativa os períodos de atividade e inatividade de cada CPE, evitando suposições rígidas baseadas apenas no calendário.

A análise evidencia que:

- existem CPE com forte dependência de atividade humana, apresentando diferenças claras entre períodos de utilização e não utilização;
- outros CPE apresentam comportamento mais estável, sugerindo funcionamento contínuo;
- a magnitude da diferença entre os dois estados varia significativamente entre CPE, refletindo diferentes tipologias de consumo.

Esta distinção constitui uma forma de contextualização dos dados, permitindo enriquecer a representação das séries temporais.

Do ponto de vista da deteção de anomalias, esta abordagem é particularmente relevante, uma vez que comportamentos considerados normais em períodos de utilização podem ser anómalos em períodos de não utilização, e vice-versa.

Adicionalmente, esta separação poderá ser explorada na fase de extração de features, contribuindo para uma melhor caracterização dos perfis de consumo e para a melhoria do desempenho dos modelos.

# Dias Úteis VS Fim de Semana

Como abordagem inicial, é analisada a diferença de comportamento entre dias úteis e fins de semana.

Esta distinção é frequentemente utilizada em análise de séries temporais, assumindo que padrões de consumo variam em função do calendário.

No entanto, esta abordagem será posteriormente comparada com métodos baseados no comportamento real dos dados, de forma a avaliar a sua adequação.

In [ ]:
# Separação entre dias úteis e fins de semana

df_temp = df_pivot.copy()

# Dia da semana (0 = segunda, 6 = domingo)
df_temp["dia_semana"] = df_temp.index.dayofweek

# Separação
df_weekday = df_temp[df_temp["dia_semana"] < 5]
df_weekend = df_temp[df_temp["dia_semana"] >= 5]

print("Shape weekday:", df_weekday.shape)
print("Shape weekend:", df_weekend.shape)

In [ ]:
# Perfis médios (weekday vs weekend)

df_weekday_clean = df_weekday.drop(columns=["dia_semana"])
df_weekend_clean = df_weekend.drop(columns=["dia_semana"])

perfil_weekday = df_weekday_clean.groupby(df_weekday_clean.index.hour).mean()
perfil_weekend = df_weekend_clean.groupby(df_weekend_clean.index.hour).mean()

print("Shape perfil weekday:", perfil_weekday.shape)
print("Shape perfil weekend:", perfil_weekend.shape)

In [ ]:
# Vizualização por CPE

graficos_dir = exploration_dir / "perfis_weekday_vs_weekend"
graficos_dir.mkdir(parents=True, exist_ok=True)

for cpe in perfil_weekday.columns:

    plt.figure(figsize=(8, 4))
    plt.plot(perfil_weekday.index, perfil_weekday[cpe], label="Semana")
    plt.plot(perfil_weekend.index, perfil_weekend[cpe], label="Fim de semana")
    plt.title(f"{cpe} - Semana vs Fim de semana")
    plt.xlabel("Hora do dia")
    plt.ylabel("PotActiva média")
    plt.xticks(range(0, 24))
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(graficos_dir / f"{cpe}_weekday_vs_weekend.png", dpi=150)
    plt.show()
    plt.close()

print(f"Gráficos guardados em: {graficos_dir}")

In [ ]:
# Diferença média entre contextos

diff_media = (perfil_weekday - perfil_weekend).abs().mean()
diff_media = diff_media.sort_values(ascending=False)

print("CPE com maior diferença entre semana e fim de semana:")
display(diff_media.head(10))

## Observações

A comparação entre dias úteis e fins de semana evidencia diferenças no comportamento de consumo para alguns CPE, nomeadamente nos casos em que existe atividade predominantemente associada a horários laborais.

No entanto, esta abordagem apresenta limitações importantes:

- assume padrões uniformes de funcionamento baseados no calendário;
- não considera especificidades de cada ponto de consumo;
- pode não capturar corretamente comportamentos atípicos ou regimes de funcionamento não convencionais.

Observa-se que, embora alguns CPE apresentem diferenças significativas entre semana e fim de semana, outros mantêm padrões semelhantes, sugerindo funcionamento contínuo ou independente do calendário.

Desta forma, esta análise deve ser interpretada como uma baseline simplificada.

No contexto deste trabalho, a distinção baseada no comportamento real de consumo (utilização vs não utilização) revela-se mais adequada, uma vez que permite adaptar a análise às características específicas de cada CPE.

Consequentemente, a abordagem baseada no calendário é utilizada apenas como referência, sendo complementada por métodos mais flexíveis nas etapas seguintes.

# Períodos do ano e dias especiais

Para além dos padrões diários, o consumo energético pode ser influenciado por fatores de longo prazo, como a sazonalidade e eventos específicos do calendário.

Nesta secção, são analisadas variações no comportamento dos CPE em diferentes períodos do ano, incluindo:
- diferenças mensais;
- padrões sazonais (estações do ano);
- dias especiais, como feriados e eventos pontuais.

A incorporação destes fatores permite enriquecer a interpretação dos dados e introduzir contexto adicional relevante para a deteção de anomalias.

In [ ]:
# Componentes temporais adicionais

df_contexto["mes"] = df_contexto["tstamp"].dt.month
df_contexto["ano"] = df_contexto["tstamp"].dt.year

A análise mensal permite identificar variações no consumo ao longo do ano, possibilitando a deteção de padrões sazonais associados a fatores como condições climatéricas ou alterações nos regimes de utilização.

In [ ]:
# Perfis por mês

perfil_mes = (
    df_contexto
    .groupby(["CPE", "mes", "hora"])["PotActiva"]
    .mean()
    .reset_index()
)

perfil_mes.head()

In [ ]:
# Visualização global por mês

perfil_mes_global = (
    df_contexto
    .groupby(["mes", "hora"])["PotActiva"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 5))

for mes in sorted(perfil_mes_global["mes"].unique()):
    df_mes = perfil_mes_global[perfil_mes_global["mes"] == mes]
    plt.plot(df_mes["hora"], df_mes["PotActiva"], label=f"Mês {mes}")

plt.title("Perfis médios por mês")
plt.xlabel("Hora do dia")
plt.ylabel("PotActiva média")
plt.xticks(range(0, 24))
plt.legend(ncol=2, fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Os perfis médios por mês evidenciam diferenças no comportamento do consumo ao longo do ano, sugerindo a presença de efeitos sazonais.

Estas variações podem estar associadas a fatores externos, como temperatura, luminosidade ou padrões de ocupação, que influenciam diretamente o consumo energético.

Para simplificar a análise sazonal, os meses são agrupados em estações do ano, permitindo observar padrões agregados de consumo em diferentes períodos climáticos.

In [ ]:
# Estações do ano

# Função para mapear mês -> estação
def get_estacao(mes):
    if mes in [12, 1, 2]:
        return "inverno"
    elif mes in [3, 4, 5]:
        return "primavera"
    elif mes in [6, 7, 8]:
        return "verao"
    else:
        return "outono"

df_contexto["estacao"] = df_contexto["mes"].apply(get_estacao)

In [ ]:
# Perfis por estação

perfil_estacao = (
    df_contexto
    .groupby(["CPE", "estacao", "hora"])["PotActiva"]
    .mean()
    .reset_index()
)

In [ ]:
# Visualização global por estação

perfil_estacao_global = (
    df_contexto
    .groupby(["estacao", "hora"])["PotActiva"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 5))

for estacao in perfil_estacao_global["estacao"].unique():
    df_est = perfil_estacao_global[perfil_estacao_global["estacao"] == estacao]
    plt.plot(df_est["hora"], df_est["PotActiva"], label=estacao)

plt.title("Perfis médios por estação do ano")
plt.xlabel("Hora do dia")
plt.ylabel("PotActiva média")
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Para além da sazonalidade regular, determinados dias apresentam comportamentos atípicos que não são capturados pelos padrões médios.

Nesta análise, são considerados:
- feriados;
- eventos específicos (ex: apagões).

A identificação destes dias permite capturar contextos excecionais que podem influenciar significativamente o consumo.

In [ ]:
# Dias especiais

df_contexto["tipo_dia_especial"] = "normal"

In [ ]:
# Feriados

feriados = pd.to_datetime([
    "2023-04-25", "2023-05-01", "2023-06-10", "2023-10-05", "2023-12-01",
    "2024-04-25", "2024-05-01", "2024-06-10", "2024-10-05", "2024-12-01",
    "2025-04-25", "2025-05-01"
]).date

df_contexto.loc[df_contexto["data"].isin(feriados), "tipo_dia_especial"] = "feriado"

In [ ]:
# Dia do Apagão

df_contexto.loc[
    df_contexto["data"] == pd.to_datetime("2025-04-28").date(),
    "tipo_dia_especial"
] = "apagao_28_abril_2025"

In [ ]:
# Perfil para dias especiais

perfil_especial = (
    df_contexto[df_contexto["tipo_dia_especial"] != "normal"]
    .groupby(["CPE", "tipo_dia_especial", "hora"])["PotActiva"]
    .mean()
    .reset_index()
)

In [ ]:
# Resumo dos dias especiais

print("Tipos de dias especiais identificados:")
display(df_contexto["tipo_dia_especial"].value_counts())

print("\nExemplo de perfis especiais:")
display(perfil_especial.head())

## Observações

A análise realizada evidencia que o consumo energético apresenta variações relevantes associadas ao calendário.

Em particular:

- observam-se diferenças ao nível mensal e sazonal, indicando a presença de padrões de longo prazo;
- os perfis por estação revelam comportamentos distintos, possivelmente associados a fatores climatéricos e de utilização;
- dias especiais, como feriados ou eventos pontuais, introduzem desvios face ao comportamento típico.

Estes resultados demonstram que o consumo energético não depende apenas de padrões diários, mas também de fatores contextuais mais amplos.

No contexto da deteção de anomalias, esta distinção é fundamental, uma vez que determinados comportamentos podem ser considerados normais apenas em contextos específicos.

Assim, a incorporação de informação temporal e contextual constitui um elemento chave para a construção de modelos mais robustos e interpretáveis.

# Consumo diário e mensal por CPE

Para complementar a análise dos perfis horários, é também importante observar o consumo energético em escalas temporais mais agregadas.

Nesta secção, o consumo é analisado ao nível diário e mensal, permitindo identificar:
- flutuações de curto prazo;
- tendências de médio prazo;
- diferenças de variabilidade entre CPE.

Esta perspetiva complementa a análise anterior e contribui para uma caracterização mais abrangente dos perfis de consumo.

In [ ]:
# Consumo diário

consumo_diario = (
    df_pivot
    .resample("D")
    .sum()
)

consumo_diario.head()

In [ ]:
# Vizualização por CPE

todos_cpes = consumo_diario.columns.tolist()

graficos_dir = exploration_dir / "consumo_diario_cpe"
graficos_dir.mkdir(parents=True, exist_ok=True)

for cpe in todos_cpes:

    plt.figure(figsize=(12, 4))
    plt.plot(consumo_diario.index, consumo_diario[cpe])
    plt.title(f"Consumo diário - {cpe}")
    plt.xlabel("Tempo")
    plt.ylabel("Consumo diário")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(graficos_dir / f"{cpe}_consumo_diario.png", dpi=150)
    plt.show()
    plt.close()

print(f"Gráficos guardados em: {graficos_dir}")

In [ ]:
# Consumo mensal

consumo_mensal = (
    df_pivot
    .resample("ME")
    .sum()
)

consumo_mensal.head()

In [ ]:
# Vizualização por CPE

todos_cpes = consumo_mensal.columns.tolist()

graficos_dir = exploration_dir / "consumo_mensal_cpe"
graficos_dir.mkdir(parents=True, exist_ok=True)

for cpe in todos_cpes:

    plt.figure(figsize=(10, 4))
    plt.plot(consumo_mensal.index, consumo_mensal[cpe], marker="o")
    plt.title(f"Consumo mensal - {cpe}")
    plt.xlabel("Tempo")
    plt.ylabel("Consumo mensal")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(graficos_dir / f"{cpe}_consumo_mensal.png", dpi=150)
    plt.show()
    plt.close()

print(f"Gráficos guardados em: {graficos_dir}")

In [ ]:
# Variabilidade diária

std_diario = consumo_diario.std()
mean_diario = consumo_diario.mean()

## Observações

A análise do consumo diário e mensal permite observar o comportamento energético em diferentes escalas temporais, complementando a análise horária previamente realizada.

Em particular:

- o consumo diário evidencia flutuações associadas a padrões de utilização e possíveis irregularidades;
- o consumo mensal permite observar tendências mais estáveis e potenciais efeitos sazonais;
- a variabilidade entre CPE confirma a existência de diferentes perfis de consumo, tanto em intensidade como em regularidade.

Esta análise reforça a heterogeneidade do dataset e evidencia a necessidade de caracterizar cada CPE de forma individual.

Do ponto de vista metodológico, estas escalas agregadas são também relevantes para a extração de features, uma vez que permitem quantificar estabilidade, variabilidade e tendência temporal.

In [ ]:
# Guardar outputs intermédios

intermediate_dir = exploration_dir / "intermediate"
intermediate_dir.mkdir(parents=True, exist_ok=True)

df_pivot.to_csv(intermediate_dir / "df_pivot.csv")
perfil_horario.to_csv(intermediate_dir / "perfil_horario.csv")
perfil_utilizacao.to_csv(intermediate_dir / "perfil_utilizacao.csv", index=False)
consumo_diario_cpe.to_csv(intermediate_dir / "consumo_diario_cpe.csv", index=False)
perfil_weekday.to_csv(intermediate_dir / "perfil_weekday.csv")
perfil_weekend.to_csv(intermediate_dir / "perfil_weekend.csv")
consumo_diario.to_csv(intermediate_dir / "consumo_diario.csv")
consumo_mensal.to_csv(intermediate_dir / "consumo_mensal.csv")

print(f"Outputs intermédios guardados em: {intermediate_dir}")

# Conclusões da exploração do dataset

A análise exploratória realizada permitiu caracterizar os padrões de consumo energético dos diferentes pontos de entrega (CPE) sob várias perspetivas: estrutura temporal, contexto de utilização, sazonalidade e consumo agregado.

De forma global, os principais aspetos observados foram:

- existência de elevada heterogeneidade entre CPE;
- presença de padrões horários distintos, compatíveis com diferentes tipologias de utilização;
- variabilidade relevante entre dias reais, que não é totalmente capturada por perfis médios;
- influência do contexto de utilização, do calendário e da sazonalidade no comportamento do consumo;
- existência de diferenças significativas de intensidade e variabilidade entre séries.

Estes resultados confirmam que o comportamento energético depende fortemente do contexto temporal e funcional de cada ponto de consumo.

Assim, a exploração realizada fornece uma base sólida para as etapas seguintes do trabalho, nomeadamente:
- extração de features representativas;
- agrupamento de CPE por semelhança comportamental;
- desenvolvimento de abordagens de deteção de anomalias mais robustas e contextualizadas.